In [ ]:
import sys
print("Path agregado")

In [ ]:
import os
import pandas as pd
import ee
import geemap
from datetime import datetime, timedelta
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection

# Autenticación e inicialización de Earth Engine
ee.Authenticate()
ee.Initialize(project='TU-PROYECTO-GEE')

# Configuración
satelite = 'COPERNICUS/S2_SR_HARMONIZED'  # Sentinel-2 Surface Reflectance Harmonized
interDate = '2026-10-05'  # Fecha de interés (ajustable)
value_cl = 25 # Máximo porcentaje de nubes
shapefile_path = r"C:\ruta\a\tu\area_de_interes.shp"
buffer_distance = 5000  # Buffer en metros

def shapefile_to_ee_polygon(shapefile_path, buffer_distance=1000):
    try:
        gdf = gpd.read_file(shapefile_path)
        if gdf.crs is None:
            raise ValueError("El shapefile no tiene un sistema de coordenadas definido.")
        gdf = gdf.to_crs('EPSG:32719')  # UTM zona 19S
        gdf['geometry'] = gdf.geometry.buffer(buffer_distance)
        gdf = gdf.to_crs('EPSG:4326')
        polygon = gdf.geometry.union_all()
        if isinstance(polygon, Polygon):
            coords = list(polygon.exterior.coords)
            return ee.Geometry.Polygon([coords])
        elif isinstance(polygon, MultiPolygon):
            multipolygon_coords = []
            for part in polygon.geoms:
                multipolygon_coords.append([list(part.exterior.coords)])
            return ee.Geometry.MultiPolygon(multipolygon_coords)
        elif isinstance(polygon, GeometryCollection):
            multipolygon_coords = []
            for geom in polygon.geoms:
                if isinstance(geom, Polygon):
                    multipolygon_coords.append([list(geom.exterior.coords)])
                elif isinstance(geom, MultiPolygon):
                    for part in geom.geoms:
                        multipolygon_coords.append([list(part.exterior.coords)])
            return ee.Geometry.MultiPolygon(multipolygon_coords)
        else:
            raise ValueError("La geometría no es válida.")
    except Exception as e:
        print(f"Error inesperado: {e}")
        return None

# Obtener AOI
AOI = shapefile_to_ee_polygon(shapefile_path, buffer_distance)

if AOI:
    interDate = datetime.strptime(interDate, '%Y-%m-%d')
    startDate = interDate - timedelta(days=365)
    endDate = interDate + timedelta(days=1)
    print(f"Buscando imágenes entre {startDate.date()} y {endDate.date()}...")

    # Filtrar colección Sentinel-2
    Collection_S2 = (
        ee.ImageCollection(satelite)
        .filterDate(startDate.strftime('%Y-%m-%d'), endDate.strftime('%Y-%m-%d'))
        .filterBounds(AOI)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', value_cl))
    )

    n = Collection_S2.size().getInfo()
    if n == 0:
        raise ValueError("No se encontraron imágenes Sentinel-2 en ese período.")
    
    print(f'Total de imágenes encontradas: {n}')

    # Obtener info de fechas y nubes
    image_info = Collection_S2.aggregate_array('system:time_start').getInfo()
    cloud_coverage = Collection_S2.aggregate_array('CLOUDY_PIXEL_PERCENTAGE').getInfo()

    df = pd.DataFrame({
        'Fecha': [datetime.utcfromtimestamp(date / 1000).strftime('%Y-%m-%d') for date in image_info],
        'Porcentaje': cloud_coverage
    })

    df_agrupado = df.groupby('Fecha', as_index=False)['Porcentaje'].mean()
    print(df_agrupado)

    # Crear el mapa
    Map = geemap.Map()

    for fecha in df_agrupado['Fecha']:
        next_day = (datetime.strptime(fecha, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')
        Collection_fecha = (
            ee.ImageCollection(satelite)
            .filterDate(fecha, next_day)
            .filterBounds(AOI)
        )

        medianpixels = Collection_fecha.median().clip(AOI).divide(10000)  # SR escala S2

        stats = medianpixels.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
            geometry=AOI,
            scale=10,
            maxPixels=1e13
        ).getInfo()

        try:
            vis_params = {
                'bands': ['B4', 'B3', 'B2'],  # RGB
                'min': min(stats[f'{band}_mean'] - stats[f'{band}_stdDev'] for band in ['B4', 'B8', 'B3']),
                'max': max(stats[f'{band}_mean'] + stats[f'{band}_stdDev'] for band in ['B4', 'B8', 'B3']),
                'gamma': 1.8
            }

            Map.addLayer(medianpixels, vis_params, f"Sentinel2_{fecha}")
            print(f"✅ Agregada imagen de {fecha} al mapa.")
        except Exception as e:
            print(f"⚠️ No se pudo visualizar la imagen de {fecha}: {e}")

    Map
    print("✅ Visualización de todas las fechas completada.")



In [ ]:
import os
import pandas as pd
import ee
import geemap
from datetime import datetime, timedelta
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection

# =========================================================
# AUTENTICACIÓN E INICIALIZACIÓN
# =========================================================
ee.Authenticate()
ee.Initialize(project='TU-PROYECTO-GEE')

# =========================================================
# CONFIGURACIÓN
# =========================================================
satelite = 'COPERNICUS/S2_SR_HARMONIZED'
interDate = '2024-12-30'
value_cl = 100
shapefile_path = r"C:\ruta\a\tu\area_de_interes.shp"
buffer_distance = 5000

# =========================================================
# FUNCIÓN: SHP → AOI Earth Engine
# =========================================================
def shapefile_to_ee_polygon(shapefile_path, buffer_distance=1000):
    try:
        gdf = gpd.read_file(shapefile_path)

        if gdf.crs is None:
            raise ValueError("El shapefile no tiene sistema de coordenadas.")

        gdf = gdf.to_crs('EPSG:32719')
        gdf['geometry'] = gdf.geometry.buffer(buffer_distance)
        gdf = gdf.to_crs('EPSG:4326')

        polygon = gdf.geometry.union_all()

        if isinstance(polygon, Polygon):
            coords = list(polygon.exterior.coords)
            return ee.Geometry.Polygon([coords])

        elif isinstance(polygon, MultiPolygon):
            multipolygon_coords = []
            for part in polygon.geoms:
                multipolygon_coords.append([list(part.exterior.coords)])
            return ee.Geometry.MultiPolygon(multipolygon_coords)

        elif isinstance(polygon, GeometryCollection):
            multipolygon_coords = []
            for geom in polygon.geoms:
                if isinstance(geom, Polygon):
                    multipolygon_coords.append([list(geom.exterior.coords)])
                elif isinstance(geom, MultiPolygon):
                    for part in geom.geoms:
                        multipolygon_coords.append([list(part.exterior.coords)])
            return ee.Geometry.MultiPolygon(multipolygon_coords)

        else:
            raise ValueError("Geometría no válida.")

    except Exception as e:
        print(f"Error inesperado: {e}")
        return None

# =========================================================
# OBTENER AOI
# =========================================================
AOI = shapefile_to_ee_polygon(shapefile_path, buffer_distance)

if AOI:

    # Crear carpeta de salida
    output_folder = os.path.dirname(shapefile_path)
    output_folder = os.path.join(output_folder, "Sentinel_descargas")
    os.makedirs(output_folder, exist_ok=True)

    print(f"Las imágenes se guardarán en: {output_folder}")

    # Definir fechas
    interDate = datetime.strptime(interDate, '%Y-%m-%d')
    startDate = interDate - timedelta(days=10)
    endDate = interDate + timedelta(days=1)

    print(f"Buscando imágenes entre {startDate.date()} y {endDate.date()}...")

    # Filtrar colección
    Collection_S2 = (
        ee.ImageCollection(satelite)
        .filterDate(startDate.strftime('%Y-%m-%d'), endDate.strftime('%Y-%m-%d'))
        .filterBounds(AOI)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', value_cl))
    )

    n = Collection_S2.size().getInfo()

    if n == 0:
        raise ValueError("No se encontraron imágenes Sentinel-2.")

    print(f'Total de imágenes encontradas: {n}')

    # Obtener fechas y nubosidad
    image_info = Collection_S2.aggregate_array('system:time_start').getInfo()
    cloud_coverage = Collection_S2.aggregate_array('CLOUDY_PIXEL_PERCENTAGE').getInfo()

    df = pd.DataFrame({
        'Fecha': [datetime.utcfromtimestamp(date / 1000).strftime('%Y-%m-%d') for date in image_info],
        'Porcentaje': cloud_coverage
    })

    df_agrupado = df.groupby('Fecha', as_index=False)['Porcentaje'].mean()
    print(df_agrupado)

    # Crear mapa
    Map = geemap.Map()

    # =========================================================
    # BUCLE PRINCIPAL
    # =========================================================
    for fecha in df_agrupado['Fecha']:

        print(f"\nProcesando fecha: {fecha}")

        next_day = (datetime.strptime(fecha, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')

        Collection_fecha = (
            ee.ImageCollection(satelite)
            .filterDate(fecha, next_day)
            .filterBounds(AOI)
        )

        medianpixels = Collection_fecha.median().clip(AOI).divide(10000)

        # Estadísticas
        stats = medianpixels.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
            geometry=AOI,
            scale=10,
            maxPixels=1e13
        ).getInfo()

        try:
            vis_params = {
                'bands': ['B4', 'B8', 'B3'],
                'min': min(stats[f'{band}_mean'] - stats[f'{band}_stdDev'] for band in ['B4', 'B8', 'B3']),
                'max': max(stats[f'{band}_mean'] + stats[f'{band}_stdDev'] for band in ['B4', 'B8', 'B3']),
                'gamma': 1.8
            }

            # Agregar al mapa
            Map.addLayer(medianpixels, vis_params, f"Sentinel2_{fecha}")
            print(f" Imagen agregada al mapa.")

            # =====================================================
            # DESCARGA AUTOMÁTICA
            # =====================================================
            output_path = os.path.join(output_folder, f"Sentinel2_{fecha}.tif")

            geemap.ee_export_image(
                medianpixels,
                filename=output_path,
                scale=10,
                region=AOI,
                file_per_band=False
            )

            print(f" Imagen descargada en: {output_path}")

        except Exception as e:
            print(f" Error en {fecha}: {e}")

    Map
    print("\n Proceso completado. Todas las imágenes fueron visualizadas y descargadas.")